# 02 — Baseline end-to-end (braindecode)
**EEGNet, ShallowFBCSPNet, Deep4Net** su segnale grezzo, subject-dependent su 15 soggetti.
Early stopping sul Validation set, valutazione sul Test set (true label da answer sheet).

> Metti `use_wandb=True` dopo `pip install wandb && wandb login` per loggare su W&B (entity `uras-daniele22-politecnico-di-milano`, project `miralis-imagined-speech`, tag `track3`).

In [ ]:
# --- setup: rende importabili i moduli track3_*.py ---
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, matplotlib.pyplot as plt
import track3_config as C, track3_io as io, track3_preproc as P
print(C.summary())
assert C.DATA_ROOT is not None, C._no_data_msg()
import track3_train as T
import track3_models as M
device=C.get_device(); print('device:', device)

## 1. EEGNet su tutti i soggetti
⚠️ Sulla GPU della VM è veloce; su CPU può richiedere qualche minuto per modello.

In [ ]:
df_eeg, res_eeg = T.run_subject_dependent(
    'eegnet', use_wandb=False,
    train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_eeg, 'eegnet'); df_eeg

In [ ]:
T.plot_per_subject(df_eeg, 'eegnet'); plt.show()
T.plot_confusion(res_eeg, model_name='eegnet'); plt.show()

## 2. ShallowFBCSPNet

In [ ]:
df_sh, res_sh = T.run_subject_dependent(
    'shallow', train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_sh, 'shallow')
T.plot_per_subject(df_sh, 'shallow'); plt.show()

## 3. Deep4Net

In [ ]:
df_d4, res_d4 = T.run_subject_dependent(
    'deep4', train_kwargs=dict(epochs=200, patience=30, lr=1e-3, batch_size=32))
T.save_metrics(df_d4, 'deep4')
T.plot_per_subject(df_d4, 'deep4'); plt.show()

## 4. Riepilogo baseline

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'eegnet':  df_eeg.test_acc, 'shallow': df_sh.test_acc, 'deep4': df_d4.test_acc,
}).describe().loc[['mean','std','min','max']]
print('chance =', C.CHANCE_LEVEL); summary